# Data loading and preprocessing

## Dataset loading

In [25]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split

# For type hinting, which is good practice
from torch.utils.data import Dataset
from typing import Tuple

def load_fashion_data() -> Tuple[Dataset, Dataset, Dataset]:
    """
    Loads the Fashion-MNIST dataset and splits the data into train, val, test.

    This function loads the Fashion-MNIST training and test sets from
    torchvision. It then splits the original training set into a new,
    smaller training set and a validation set (90/10 ratio).

    Returns:
        Tuple[Dataset, Dataset, Dataset]: A tuple containing the training,
        validation, and test datasets as torch.utils.data.Dataset objects.
    """
    # Use a basic transform to convert images to tensors.
    # More complex transforms will be applied later in the custom Dataset class.
    # converts a PIL Image or numpy.ndarray to tensor.
    transform = transforms.ToTensor()

    # 1. Load the full original training data
    full_train_dataset = datasets.FashionMNIST(
        root='./data',
        train=True,
        download=True,
        transform=transform
    )

    # 2. Load the test data
    test_dataset = datasets.FashionMNIST(
        root='./data',
        train=False,
        download=True,
        transform=transform
    )

    # 3. Calculate split sizes
    dataset_size = len(full_train_dataset)
    train_size = int(0.9 * dataset_size)
    val_size = dataset_size - train_size

    # 4. Split the full training dataset into training and validation subsets
    # Using a fixed generator seed for reproducibility of the split
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=generator
    )

    return train_dataset, val_dataset, test_dataset

# # Example of how to call the function and verify the output
# if __name__ == '__main__':
#     train_data, val_data, test_data = load_fashion_data()

#     print(f"Fashion-MNIST Dataset Summary:")
#     print(f"  - Total Original Training Images: {len(train_data) + len(val_data)}")
#     print(f"  - Split Training Set Size:      {len(train_data)}")
#     print(f"  - Split Validation Set Size:    {len(val_data)}")
#     print(f"  - Test Set Size:                {len(test_data)}")

## Custom Dataset Class

In [26]:
import torch
from torch.utils.data import Dataset, Subset
from torchvision import transforms
from typing import Tuple, Any

def calculate_mean_std(dataset: Dataset) -> Tuple[Tuple[float, ...], Tuple[float, ...]]:
    """
    Calculate the mean and standard deviation of a dataset.
    
    Args:
        dataset: A PyTorch dataset containing images
        
    Returns:
        Tuple containing mean and std as tuples
    """
    # Stack all images into a single tensor
    # We'll iterate through the dataset and collect all pixel values
    all_pixels = []
    
    for img, _ in dataset:
        # img is already a tensor from ToTensor() transform
        all_pixels.append(img)
    
    # Stack all images: shape will be [N, C, H, W]
    all_pixels = torch.stack(all_pixels)
    
    # Calculate mean and std across all dimensions except the channel dimension
    # Compute per-channel statistics
    mean = all_pixels.mean(dim=[0, 2, 3])  # Average over batch, height, width
    std = all_pixels.std(dim=[0, 2, 3])    # Std dev over batch, height, width
    
    return tuple(mean.tolist()), tuple(std.tolist())

# # Load the training data to calculate statistics
# train_dataset, val_dataset, test_dataset = load_fashion_data()

# # Calculate mean and std from the training set
# print("Calculating mean and standard deviation from training data...")
# FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

# print(f"Calculated Mean: {FASHION_MNIST_MEAN}")
# print(f"Calculated Std:  {FASHION_MNIST_STD}")

In [27]:
# # Load the training data to calculate statistics
# train_dataset, val_dataset, test_dataset = load_fashion_data()

# # Calculate mean and std from the training set
# print("Calculating mean and standard deviation from training data...")
# FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

# print(f"Calculated Mean: {FASHION_MNIST_MEAN}")
# print(f"Calculated Std:  {FASHION_MNIST_STD}")

In [28]:
class FashionMNISTDataset(Dataset):
    """
    Custom Dataset wrapper for Fashion-MNIST that returns:
    - Normalized image tensor
    - Class label
    - Ink target (normalized pixel intensity)
    
    Supports optional data augmentation for training.
    """
    
    def __init__(
        self, 
        base_dataset: Dataset,
        mean: Tuple[float, ...],
        std: Tuple[float, ...],
        augment: bool = False
    ):
        """
        Initialize the custom Fashion-MNIST dataset.
        
        Args:
            base_dataset: The underlying Fashion-MNIST dataset (can be a Subset)
            mean: Mean values for normalization (per channel)
            std: Standard deviation values for normalization (per channel)
            augment: Whether to apply data augmentation (for training only)
        """
        self.base_dataset = base_dataset
        self.mean = mean
        self.std = std
        self.augment = augment
        
        # Create normalization transform
        self.normalize = transforms.Normalize(mean=mean, std=std)
        
        # Create augmentation transforms (light augmentations)
        if self.augment:
            self.augmentation = transforms.Compose([
                # Random crop: crop to 28x28 from a 32x32 padded image
                # This creates slight translations
                transforms.RandomCrop(28, padding=4),
                
                # Small random rotation (up to 10 degrees)
                transforms.RandomRotation(degrees=10),
                
                # Random horizontal flip (50% chance)
                # Note: This is reasonable for most Fashion-MNIST items
                transforms.RandomHorizontalFlip(p=0.5),
            ])
        
    def __len__(self) -> int:
        """Return the total number of samples in the dataset."""
        return len(self.base_dataset)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, float]:
        """
        Get a single item from the dataset.
        
        Args:
            idx: Index of the item to retrieve
            
        Returns:
            Tuple containing:
                - image: Normalized (and optionally augmented) image tensor of shape [C, H, W]
                - class_label: Integer label in range [0, 9]
                - ink_target: Normalized pixel intensity (average pixel value)
        """
        # Get the original image and label from the base dataset
        image, class_label = self.base_dataset[idx]
        
        # Calculate ink target BEFORE any transformations
        # Ink is the average pixel value (mean intensity) of the original image
        # image is already a tensor from ToTensor() with values in [0, 1]
        ink_target = image.mean().item()
        
        # Apply augmentation if enabled (training set only)
        if self.augment:
            image = self.augmentation(image)
        
        # Normalize the image using the dataset mean and std
        normalized_image = self.normalize(image)
        
        return normalized_image, class_label, ink_target

## Data Augmentation

We apply light augmentations to the training set only to improve generalization:
- **Random Crop**: Crop 28x28 from a 32x32 padded image (creates small translations)
- **Random Rotation**: Small rotations up to ±10 degrees
- **Random Horizontal Flip**: 50% chance of horizontal flip

Validation and test sets remain unaltered for fair evaluation.

In [29]:
# # Load the training data to calculate statistics
# train_dataset, val_dataset, test_dataset = load_fashion_data()

# # Calculate mean and std from the training set
# print("Calculating mean and standard deviation from training data...")
# FASHION_MNIST_MEAN, FASHION_MNIST_STD = calculate_mean_std(train_dataset)

# print(f"Calculated Mean: {FASHION_MNIST_MEAN}")
# print(f"Calculated Std:  {FASHION_MNIST_STD}")

# print("\n" + "="*60)

# # Create custom dataset instances
# # Training set WITH augmentation
# train_custom = FashionMNISTDataset(
#     train_dataset, 
#     FASHION_MNIST_MEAN, 
#     FASHION_MNIST_STD,
#     augment=True  # Enable augmentation for training
# )

# # Validation and test sets WITHOUT augmentation
# val_custom = FashionMNISTDataset(
#     val_dataset, 
#     FASHION_MNIST_MEAN, 
#     FASHION_MNIST_STD,
#     augment=False  # No augmentation for validation
# )

# test_custom = FashionMNISTDataset(
#     test_dataset, 
#     FASHION_MNIST_MEAN, 
#     FASHION_MNIST_STD,
#     augment=False  # No augmentation for test
# )

# print("\nCustom datasets created successfully!")
# print(f"Training set size: {len(train_custom)} (with augmentation)")
# print(f"Validation set size: {len(val_custom)} (no augmentation)")
# print(f"Test set size: {len(test_custom)} (no augmentation)")

# # Test the dataset by getting one sample
# sample_image, sample_label, sample_ink = train_custom[0]
# print(f"\nSample from training dataset:")
# print(f"  Image shape: {sample_image.shape}")
# print(f"  Image dtype: {sample_image.dtype}")
# print(f"  Class label: {sample_label}")
# print(f"  Ink target: {sample_ink:.4f}")
# print(f"  Image min value: {sample_image.min():.4f}")
# print(f"  Image max value: {sample_image.max():.4f}")

In [30]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Visualize the effect of augmentation
# # Get the same sample multiple times to see different augmentations
# fig, axes = plt.subplots(2, 5, figsize=(15, 6))
# fig.suptitle('Effect of Data Augmentation on Training Set', fontsize=16, fontweight='bold')

# # Get one sample index
# sample_idx = 0

# # Row 1: Original (no augmentation)
# # axes[0, 0].set_ylabel('Without Augmentation', fontsize=12, fontweight='bold')
# # Create a non-augmented dataset temporarily for comparison

# train_no_aug = FashionMNISTDataset(
#     train_dataset, 
#     FASHION_MNIST_MEAN, 
#     FASHION_MNIST_STD,
#     augment=False
# )
# img, label, ink = train_no_aug[sample_idx]

# # Denormalize for visualization
# img_denorm = img * FASHION_MNIST_STD[0] + FASHION_MNIST_MEAN[0]
# img_denorm = img_denorm.squeeze().numpy()

# for col in range(5):
    
#     axes[0, col].imshow(img_denorm, cmap='gray')
#     axes[0, col].axis('off')
#     if col == 0:
#         axes[0, col].set_title(f'Label: {label}\nInk: {ink:.3f}', fontsize=10)
#     else:
#         axes[0, col].set_title('(Same Image)', fontsize=10)

# # Row 2: With augmentation (different each time)
# # axes[1, 0].set_ylabel('With Augmentation', fontsize=12, fontweight='bold')
# for col in range(5):
#     img, label, ink = train_custom[sample_idx]
    
#     # Denormalize for visualization
#     img_denorm = img * FASHION_MNIST_STD[0] + FASHION_MNIST_MEAN[0]
#     img_denorm = img_denorm.squeeze().numpy()
    
#     axes[1, col].imshow(img_denorm, cmap='gray')
#     axes[1, col].axis('off')
#     if col == 0:
#         axes[1, col].set_title(f'Label: {label}\nInk: {ink:.3f}', fontsize=10)
#     else:
#         axes[1, col].set_title('(Random Aug)', fontsize=10)

# plt.tight_layout()
# plt.show()

# print("Note: Each time you access an augmented sample, different random transformations are applied!")
# print("This helps the model generalize better by seeing variations of the same image.")

In [31]:
# # Create custom dataset instances
# train_custom = FashionMNISTDataset(train_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD)
# val_custom = FashionMNISTDataset(val_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD)
# test_custom = FashionMNISTDataset(test_dataset, FASHION_MNIST_MEAN, FASHION_MNIST_STD)

# print("Custom datasets created successfully!")
# print(f"Training set size: {len(train_custom)}")
# print(f"Validation set size: {len(val_custom)}")
# print(f"Test set size: {len(test_custom)}")

# # Test the dataset by getting one sample
# sample_image, sample_label, sample_ink = train_custom[0]
# print(f"\nSample from dataset:")
# print(f"  Image shape: {sample_image.shape}")
# print(f"  Image dtype: {sample_image.dtype}")
# print(f"  Class label: {sample_label}")
# print(f"  Ink target: {sample_ink:.4f}")
# print(f"  Image min value: {sample_image.min():.4f}")
# print(f"  Image max value: {sample_image.max():.4f}")